In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.experiments import ExperimentConfig, run_experiment

In [3]:
cfg_tuned = ExperimentConfig(
    name="fuzzy_wine_tuned",
    datasets=["wine"],
    settings=["fixed", "variable"],
    methods=["Fuzzy-ACE"],
    n_runs=10,
    sampling_ratio=1.0,
    alpha_1=0.6,
    fuzzy_ace_m_range=(1.2, 1.8),
    n_jobs=-1,
)
raw_tuned = run_experiment(cfg_tuned, resume=False, verbose=True)

[fuzzy_wine_tuned] 20 runs to do (0 already stored) -> /media/marta/Dane/Sem9/FuzzyAdaptiveClusteringEnsemble/results/fuzzy_wine_tuned
  [1/20] fixed    wine               Fuzzy-ACE run 1: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [2/20] fixed    wine               Fuzzy-ACE run 3: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [3/20] fixed    wine               Fuzzy-ACE run 2: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [4/20] fixed    wine               Fuzzy-ACE run 8: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [5/20] fixed    wine               Fuzzy-ACE run 0: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [6/20] fixed    wine               Fuzzy-ACE run 4: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [7/20] fixed    wine               Fuzzy-ACE run 7: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [8/20] fixed    wine               Fuzzy-ACE run 9: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [9/20] fixed    wine               Fuzzy-ACE run 5: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [10/20] fixed    wine               Fuzzy-ACE run 6: ARI=0.8975 NMI=0.8759 (0.2s) ok
  [1

In [4]:
import itertools
DATASETS = ["wine", "glass", "ecoli"]
ALPHA_1_GRID = [0.4, 0.5, 0.6, 0.7, 0.8]
M_RANGE_GRID = [(1.2, 1.8), (1.5, 2.0), (1.5, 2.5)]

records = []
for a1, m_range in itertools.product(ALPHA_1_GRID, M_RANGE_GRID):
    tag = f"grid_a1_{a1}_m_{m_range[0]}_{m_range[1]}"
    cfg = ExperimentConfig(
        name=f"fuzzy_grid/{tag}",
        datasets=DATASETS,
        settings=["fixed", "variable"],
        methods=["Fuzzy-ACE"],
        n_runs=10,
        sampling_ratio=1.0,
        alpha_1=a1,
        fuzzy_ace_m_range=m_range,
        n_jobs=-1,
    )
    raw = run_experiment(cfg, resume=True, verbose=False)
    ok = raw[raw["status"] == "ok"]
    for (setting, ds), g in ok.groupby(["setting", "dataset"]):
        records.append({
            "alpha_1": a1,
            "m_range": str(m_range),
            "setting": setting,
            "dataset": ds,
            "ari_mean": g["ari"].mean(),
            "ari_std": g["ari"].std(),
            "nmi_mean": g["nmi"].mean(),
        })

grid_df = pd.DataFrame(records)
# Rank combinations by average ARI across the 3 datasets
summary = (
    grid_df.groupby(["setting", "alpha_1", "m_range"])
    .agg(avg_ari=("ari_mean", "mean"), avg_std=("ari_std", "mean"), avg_nmi=("nmi_mean", "mean"))
    .reset_index()
    .sort_values(["setting", "avg_ari"], ascending=[True, False])
)
display(summary)

,setting,alpha_1,m_range,avg_ari,avg_std,avg_nmi
0,fixed,0.4,"(1.2, 1.8)",0.478912,0.010704,0.583792
6,fixed,0.6,"(1.2, 1.8)",0.478456,0.009916,0.583112
9,fixed,0.7,"(1.2, 1.8)",0.478456,0.009916,0.583112
12,fixed,0.8,"(1.2, 1.8)",0.478456,0.009916,0.583112
3,fixed,0.5,"(1.2, 1.8)",0.474620,0.021914,0.580116
1,fixed,0.4,"(1.5, 2.0)",0.440268,0.072594,0.552085
4,fixed,0.5,"(1.5, 2.0)",0.440265,0.072590,0.552095
7,fixed,0.6,"(1.5, 2.0)",0.440265,0.072590,0.552095
10,fixed,0.7,"(1.5, 2.0)",0.440265,0.072590,0.552095
13,fixed,0.8,"(1.5, 2.0)",0.440265,0.072590,0.552095


In [5]:
cfg_low_m = ExperimentConfig(
    name="fuzzy_grid/grid_a1_0.8_m_1.05_1.5",
    datasets=["wine", "glass", "ecoli"],
    settings=["fixed", "variable"],
    methods=["Fuzzy-ACE"],
    n_runs=10,
    sampling_ratio=1.0,
    alpha_1=0.8,
    fuzzy_ace_m_range=(1.05, 1.5),
    n_jobs=-1,
)

raw_low = run_experiment(cfg_low_m, resume=True, verbose=False)
ok = raw_low[raw_low["status"] == "ok"]

summary_low = (
    ok.groupby(["setting", "dataset"])
    .agg(ari_mean=("ari", "mean"), ari_std=("ari", "std"), nmi_mean=("nmi", "mean"))
    .groupby("setting")
    .agg(avg_ari=("ari_mean", "mean"), avg_std=("ari_std", "mean"), avg_nmi=("nmi_mean", "mean"))
    .reset_index()
)
display(summary_low)

,setting,avg_ari,avg_std,avg_nmi
0,fixed,0.498650,0.004718,0.599044
1,variable,0.475941,0.082198,0.571873
